In [5]:
from pathlib import Path
import re
import sys

import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ============================================================
# Configuration
# ============================================================

INPUT_FILE = Path(
    r"D:/10. RQ1 DySec/Step 11 MaliciousPackagesTraces/"
    r"Installation Data/MaliciousFinalSelectedPackages -Status.xlsx"
)

OUTPUT_FILE = Path(
    r"D:/12. RQ2 eDySec/Adversarial Attacks/"
    r"high_risk_packages_107.xlsx"
)

def command_line_paths() -> list:
    """Return positional path arguments, or [] when running in a notebook."""
    try:
        get_ipython 
        return []
    except NameError:
        pass

    if "ipykernel" in Path(sys.argv[0]).name:
        return []

    return [
        argument
        for argument in sys.argv[1:]
        if not argument.startswith("-")
    ]


cli_paths = command_line_paths()

if len(cli_paths) > 0:
    INPUT_FILE = Path(cli_paths[0])
if len(cli_paths) > 1:
    OUTPUT_FILE = Path(cli_paths[1])

SHEET_NAME = "MaliciousPackageNameAndVersion"

TARGET_UNIQUE_PACKAGES = 107


SEARCH_COLUMNS = [
    "Auto Status (Python 3.12)",
    "Error Type (Python 3.12)",
    "Error Type (Python 3.*)",
    "Remarks",
]

# ============================================================
# Risk tiers
# ============================================================

RISK_TIERS = [
    (
        "Tier 1 - Sandbox / runtime anomaly",
        {
            "Shutdown PC": ["shutdown pc"],
            "Infinity waiting": ["infinity waiting"],
            "Freeze": ["freeze"],
            "Version loop": ["version loop"],
        },
    ),
    (
        "Tier 2 - Corrupt or malformed artifact",
        {
            "ReadError: empty file": [
                "readerror: empty file",
                "read error: empty file",
                "empty file",
            ],
            "Compressed file ended before the end": [
                "compressed file ended before the end",
                "compressed file ended",
            ],
            "Not a gzip file problem": ["not a gzip file"],
            "File name problem": [
                "file name problem",
                "name problem",
            ],
        },
    ),
    (
        "Tier 3 - Install-process failure",
        {
            "Legacy-install-failure": [
                "legacy install failure",
                "legacy-install-failure",
            ],
            "No module named problem": [
                "module not found",
                "no module named",
            ],
        },
    ),
    (
        "Tier 4 - Dependency resolution failure",
        {
            "Not find a version that satisfies problem": [
                "could not find a version",
            ],
            "Matching distribution found problem": [
                "no matching distribution",
            ],
        },
    ),
    (
        "Tier 5 - Environment constraint failure",
        {
            "Requires a different Python problem": [
                "requires a different python",
            ],
        },
    ),
]

# Tie-break inside the tier that is only partially consumed.
# More version records first, then alphabetical package name.
PARTIAL_TIER_ORDER = (
    ["Version Records", "Package Name"],
    [False, True],
)


# ============================================================
# Helper functions
# ============================================================

def normalize_text(value) -> str:
    if pd.isna(value):
        return ""

    text = str(value).strip().lower()
    text = re.sub(r"[\s_-]+", " ", text)

    return text


def classify_row(row: pd.Series):
    for tier_index, (tier_name, rules) in enumerate(RISK_TIERS):
        for category, patterns in rules.items():
            for column in SEARCH_COLUMNS:
                value = normalize_text(row.get(column, ""))

                if not value:
                    continue

                for pattern in patterns:
                    if normalize_text(pattern) in value:
                        return (
                            tier_index,
                            tier_name,
                            category,
                            f"{column}: {row.get(column, '')}",
                        )

    return None


# ============================================================
# Load workbook
# ============================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found:\n{INPUT_FILE}")

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(
    INPUT_FILE,
    sheet_name=SHEET_NAME,
    engine="openpyxl",
)

print(f"Rows loaded   : {len(df):,}")
print(f"Columns       : {len(df.columns)}")

required_columns = ["Package Name", "Package Version", *SEARCH_COLUMNS]

missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise KeyError(
        "The following required columns are missing:\n"
        + "\n".join(missing_columns)
    )


# ============================================================
# Classify every row
# ============================================================

df = df.copy()

# Excel row number: header occupies row 1.
df["Source Excel Row"] = df.index + 2

classified = df.apply(classify_row, axis=1)

df["Risk Tier Index"] = classified.map(
    lambda r: r[0] if r else pd.NA
)
df["Risk Tier"] = classified.map(
    lambda r: r[1] if r else ""
)
df["High-Risk Category"] = classified.map(
    lambda r: r[2] if r else ""
)
df["Matched Evidence"] = classified.map(
    lambda r: r[3] if r else ""
)

matched = df[df["High-Risk Category"].ne("")].copy()
matched["Package Name"] = matched["Package Name"].astype(str)

print(f"Matched rows  : {len(matched):,}")


# ============================================================
# Collapse to one record per package name
# ============================================================

matched = matched.sort_values(
    ["Risk Tier Index", "Source Excel Row"],
    kind="mergesort",
)

version_counts = (
    matched.groupby("Package Name")["Package Version"]
    .nunique()
    .rename("Version Records")
)

package_level = (
    matched.drop_duplicates(subset=["Package Name"], keep="first")
    .merge(version_counts, on="Package Name", how="left")
    .reset_index(drop=True)
)

print(f"Unique names  : {len(package_level):,}")

if len(package_level) < TARGET_UNIQUE_PACKAGES:
    raise ValueError(
        f"Only {len(package_level)} unique package names are available, "
        f"but {TARGET_UNIQUE_PACKAGES} were requested. "
        "Add a further tier to RISK_TIERS."
    )


# ============================================================
# Fill the ladder until the target is reached
# ============================================================

selected_frames = []
ladder_rows = []

remaining = TARGET_UNIQUE_PACKAGES
cumulative = 0

for tier_index, (tier_name, _rules) in enumerate(RISK_TIERS):

    tier_block = package_level[
        package_level["Risk Tier Index"] == tier_index
    ]

    available = len(tier_block)

    if remaining <= 0:
        taken_block = tier_block.iloc[0:0]
        selection_note = "Not required - target already reached"

    elif available <= remaining:
        taken_block = tier_block
        selection_note = "Tier taken in full"

    else:
        columns, ascending = PARTIAL_TIER_ORDER

        taken_block = tier_block.sort_values(
            columns,
            ascending=ascending,
            kind="mergesort",
        ).head(remaining)

        selection_note = (
            "Tier partially taken to meet the target: ranked by "
            "version-record count (descending), then package name"
        )

    taken = len(taken_block)
    remaining -= taken
    cumulative += taken

    selected_frames.append(taken_block)

    ladder_rows.append(
        {
            "Risk Tier": tier_name,
            "Unique Names Available": available,
            "Unique Names Taken": taken,
            "Cumulative Selected": cumulative,
            "Selection Note": selection_note,
        }
    )

selected = (
    pd.concat(selected_frames)
    .sort_values(
        ["Risk Tier Index", "Package Name"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

assert len(selected) == TARGET_UNIQUE_PACKAGES
assert selected["Package Name"].nunique() == TARGET_UNIQUE_PACKAGES

selection_ladder = pd.DataFrame(ladder_rows)


# ============================================================
# Output tables
# ============================================================

detail_columns = [
    "Package Name",
    "Package Version",
    "Risk Tier",
    "High-Risk Category",
    "Matched Evidence",
    "Version Records",
    "Source Excel Row",
    "Prerequisites-1",
    "Other Prerequisites",
    "Installation Status",
    "Auto Status (Python 3.12)",
    "Error Type (Python 3.12)",
    "Error Type (Python 3.*)",
    "Remarks",
]

detail_columns = [c for c in detail_columns if c in selected.columns]

selected_detail = selected[detail_columns].copy()


# Every version record belonging to a selected package name.
all_versions = (
    matched[matched["Package Name"].isin(selected["Package Name"])]
    .sort_values(["Package Name", "Source Excel Row"], kind="mergesort")
    [
        [
            c
            for c in [
                "Package Name",
                "Package Version",
                "Risk Tier",
                "High-Risk Category",
                "Matched Evidence",
                "Source Excel Row",
            ]
            if c in matched.columns
        ]
    ]
    .reset_index(drop=True)
)


category_summary = (
    selected.groupby(["Risk Tier", "High-Risk Category"], as_index=False)
    .agg(
        Unique_Package_Names=("Package Name", "nunique"),
        Version_Records=("Version Records", "sum"),
    )
    .sort_values(
        ["Risk Tier", "Unique_Package_Names"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)


criteria_rows = []

for tier_index, (tier_name, rules) in enumerate(RISK_TIERS):
    for category, patterns in rules.items():
        criteria_rows.append(
            {
                "Risk Tier": tier_name,
                "High-Risk Category": category,
                "Matched Patterns": "; ".join(patterns),
                "Searched Columns": "; ".join(SEARCH_COLUMNS),
            }
        )

matching_criteria = pd.DataFrame(criteria_rows)


count_audit = pd.DataFrame(
    [
        {
            "Metric": "Rows in source sheet",
            "Count": len(df),
            "Meaning": "All package-version rows in the workbook.",
        },
        {
            "Metric": "Rows matching any risk category",
            "Count": len(matched),
            "Meaning": "Rows matching at least one configured pattern.",
        },
        {
            "Metric": "Unique package names available",
            "Count": len(package_level),
            "Meaning": "Distinct names across all five risk tiers.",
        },
        {
            "Metric": "Target unique package names",
            "Count": TARGET_UNIQUE_PACKAGES,
            "Meaning": "Requested size of the adversarial-attack set.",
        },
        {
            "Metric": "Unique package names selected",
            "Count": len(selected),
            "Meaning": "Final set, one representative record per name.",
        },
        {
            "Metric": "Version records for selected names",
            "Count": len(all_versions),
            "Meaning": "All matching versions of the selected packages.",
        },
    ]
)


# ============================================================
# Save Excel workbook
# ============================================================

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:

    selected_detail.to_excel(
        writer, sheet_name="Selected Packages", index=False
    )
    all_versions.to_excel(
        writer, sheet_name="All Versions", index=False
    )
    category_summary.to_excel(
        writer, sheet_name="Risk Summary", index=False
    )
    selection_ladder.to_excel(
        writer, sheet_name="Selection Ladder", index=False
    )
    matching_criteria.to_excel(
        writer, sheet_name="Matching Criteria", index=False
    )
    count_audit.to_excel(
        writer, sheet_name="Count Audit", index=False
    )


# ============================================================
# Format workbook
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

workbook = load_workbook(OUTPUT_FILE)

header_fill = PatternFill(fill_type="solid", fgColor="17365D")
header_font = Font(name="Arial", size=11, color="FFFFFF", bold=True)
body_font = Font(name="Arial", size=10)

for worksheet in workbook.worksheets:

    worksheet.freeze_panes = "A2"

    for row in worksheet.iter_rows():
        for cell in row:
            cell.font = body_font
            cell.alignment = Alignment(vertical="top", wrap_text=True)

    for cell in worksheet[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    for column_cells in worksheet.columns:
        column_letter = get_column_letter(column_cells[0].column)

        maximum_length = 0

        for cell in column_cells:
            value = "" if cell.value is None else str(cell.value)
            maximum_length = max(maximum_length, len(value))

        worksheet.column_dimensions[column_letter].width = min(
            maximum_length + 2, 60
        )

    worksheet.auto_filter.ref = worksheet.dimensions

workbook.save(OUTPUT_FILE)


# ============================================================
# Save CSV files
# ============================================================

csv_folder = OUTPUT_FILE.parent / "high_risk_packages_107_csv"
csv_folder.mkdir(parents=True, exist_ok=True)

selected_detail.to_csv(
    csv_folder / "selected_packages.csv", index=False, encoding="utf-8-sig"
)
all_versions.to_csv(
    csv_folder / "all_versions.csv", index=False, encoding="utf-8-sig"
)
category_summary.to_csv(
    csv_folder / "risk_summary.csv", index=False, encoding="utf-8-sig"
)
selection_ladder.to_csv(
    csv_folder / "selection_ladder.csv", index=False, encoding="utf-8-sig"
)
count_audit.to_csv(
    csv_folder / "count_audit.csv", index=False, encoding="utf-8-sig"
)


# ============================================================
# Figure: unique package names per category
# ============================================================

plot_data = (
    category_summary.groupby("High-Risk Category", as_index=False)[
        "Unique_Package_Names"
    ]
    .sum()
    .sort_values("Unique_Package_Names", ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 5.5))

bars = ax.barh(
    plot_data["High-Risk Category"],
    plot_data["Unique_Package_Names"],
    color="#17365D",
)

ax.set_xlabel("Unique package names")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.30, linewidth=0.6)
ax.set_axisbelow(True)

for bar, count in zip(bars, plot_data["Unique_Package_Names"]):
    ax.text(
        bar.get_width() + 0.4,
        bar.get_y() + bar.get_height() / 2,
        str(int(count)),
        va="center",
        ha="left",
        fontsize=9,
    )

fig.tight_layout()

fig.savefig(
    OUTPUT_FILE.parent / "high_risk_category_frequency.png",
    dpi=400,
    bbox_inches="tight",
)
fig.savefig(
    OUTPUT_FILE.parent / "high_risk_category_frequency.pdf",
    bbox_inches="tight",
)


# ============================================================
# Console summary
# ============================================================

print("\nSelection ladder")
print("-" * 78)
print(selection_ladder.to_string(index=False))

print("\nSelected packages by category")
print("-" * 78)
print(category_summary.to_string(index=False))

print(f"\nOutput workbook:\n{OUTPUT_FILE}")

Rows loaded   : 7,126
Columns       : 12
Matched rows  : 263
Unique names  : 214

Selection ladder
------------------------------------------------------------------------------
                              Risk Tier  Unique Names Available  Unique Names Taken  Cumulative Selected                                                                                          Selection Note
     Tier 1 - Sandbox / runtime anomaly                      24                  24                   24                                                                                      Tier taken in full
 Tier 2 - Corrupt or malformed artifact                       8                   8                   32                                                                                      Tier taken in full
       Tier 3 - Install-process failure                      14                  14                   46                                                                                      Tier 

## Analysis

In [7]:
from pathlib import Path
import sys
import textwrap

import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ============================================================
# Configuration
# ============================================================

CSV_FOLDER = Path("high_risk_packages_107_csv")
FIGURE_FOLDER = Path("figures_107")

def command_line_paths() -> list:
    """Return positional path arguments, or [] when running in a notebook."""
    try:
        get_ipython 
        return []
    except NameError:
        pass

    if "ipykernel" in Path(sys.argv[0]).name:
        return []

    return [
        argument
        for argument in sys.argv[1:]
        if not argument.startswith("-")
    ]


cli_paths = command_line_paths()

if len(cli_paths) > 0:
    CSV_FOLDER = Path(cli_paths[0])
if len(cli_paths) > 1:
    FIGURE_FOLDER = Path(cli_paths[1])

FIGURE_FOLDER.mkdir(parents=True, exist_ok=True)

TIER_COLORS = {
    "Tier 1": "#17365D",
    "Tier 2": "#2E75B6",
    "Tier 3": "#4EA72E",
    "Tier 4": "#ED7D31",
    "Tier 5": "#A6A6A6",
}

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 120,
    }
)


def tier_key(label: str) -> str:
    """'Tier 3 - Install-process failure' -> 'Tier 3'."""
    return str(label).split(" - ")[0].strip()


def save(fig, name: str) -> None:
    fig.savefig(
        FIGURE_FOLDER / f"{name}.png", dpi=400, bbox_inches="tight"
    )
    fig.savefig(
        FIGURE_FOLDER / f"{name}.pdf", bbox_inches="tight"
    )
    plt.close(fig)
    print(f"saved: {name}.png / {name}.pdf")


# ============================================================
# Load
# ============================================================

risk_summary = pd.read_csv(
    CSV_FOLDER / "risk_summary.csv", encoding="utf-8-sig"
)
selection_ladder = pd.read_csv(
    CSV_FOLDER / "selection_ladder.csv", encoding="utf-8-sig"
)
selected = pd.read_csv(
    CSV_FOLDER / "selected_packages.csv", encoding="utf-8-sig"
)

risk_summary["Tier"] = risk_summary["Risk Tier"].map(tier_key)
selection_ladder["Tier"] = selection_ladder["Risk Tier"].map(tier_key)
selected["Tier"] = selected["Risk Tier"].map(tier_key)


# ============================================================
# Figure 1 - unique package names per failure category
# ============================================================

data = risk_summary.sort_values(
    ["Tier", "Unique_Package_Names"], ascending=[False, True]
)

fig, ax = plt.subplots(figsize=(8.2, 5.2))

bars = ax.barh(
    data["High-Risk Category"],
    data["Unique_Package_Names"],
    color=[TIER_COLORS[t] for t in data["Tier"]],
    edgecolor="white",
    linewidth=0.6,
)

for bar, value in zip(bars, data["Unique_Package_Names"]):
    ax.text(
        bar.get_width() + 0.6,
        bar.get_y() + bar.get_height() / 2,
        str(int(value)),
        va="center",
        ha="left",
        fontsize=9,
    )

ax.set_xlabel("Unique package names")
ax.set_xlim(0, data["Unique_Package_Names"].max() * 1.14)
ax.grid(axis="x", alpha=0.30, linewidth=0.6)
ax.set_axisbelow(True)

legend_tiers = (
    risk_summary[["Tier", "Risk Tier"]]
    .drop_duplicates()
    .sort_values("Tier")
)

ax.legend(
    handles=[
        Patch(facecolor=TIER_COLORS[t], label=label)
        for t, label in zip(
            legend_tiers["Tier"], legend_tiers["Risk Tier"]
        )
    ],
    loc="upper right",
    frameon=False,
    fontsize=8.5,
)

fig.tight_layout()
save(fig, "fig1_category_frequency")


# ============================================================
# Figure 2 - selection ladder
# ============================================================

fig, ax = plt.subplots(figsize=(8.2, 4.6))

x = range(len(selection_ladder))
width = 0.62

available = ax.bar(
    x,
    selection_ladder["Unique Names Available"],
    width=width,
    color="#D9D9D9",
    edgecolor="white",
    label="Available in tier",
)

taken = ax.bar(
    x,
    selection_ladder["Unique Names Taken"],
    width=width,
    color=[TIER_COLORS[t] for t in selection_ladder["Tier"]],
    edgecolor="white",
    label="Selected",
)

ax.plot(
    x,
    selection_ladder["Cumulative Selected"],
    color="#C00000",
    marker="o",
    markersize=5,
    linewidth=1.6,
    label="Cumulative selected",
)

for xi, value in zip(x, selection_ladder["Cumulative Selected"]):
    ax.annotate(
        str(int(value)),
        (xi, value),
        textcoords="offset points",
        xytext=(0, 9),
        ha="center",
        fontsize=9,
        color="#C00000",
    )

for rect_a, rect_t in zip(available, taken):
    if rect_a.get_height() != rect_t.get_height():
        ax.text(
            rect_a.get_x() + rect_a.get_width() / 2,
            rect_a.get_height() + 2,
            str(int(rect_a.get_height())),
            ha="center",
            fontsize=8,
            color="#7F7F7F",
        )

ax.axhline(
    107,
    color="#C00000",
    linestyle="--",
    linewidth=0.9,
    alpha=0.55,
)

ax.text(
    0.35,
    109.5,
    "target = 107",
    ha="left",
    va="bottom",
    fontsize=8.5,
    color="#C00000",
)

ax.set_xticks(list(x))
ax.set_xticklabels(
    [
        textwrap.fill(label.replace(" - ", ": "), 18)
        for label in selection_ladder["Risk Tier"]
    ],
    fontsize=8,
)

ax.set_ylabel("Unique package names")
ax.set_ylim(0, 130)
ax.grid(axis="y", alpha=0.30, linewidth=0.6)
ax.set_axisbelow(True)
ax.legend(frameon=False, fontsize=8.5, loc="upper left")

fig.tight_layout()
save(fig, "fig2_selection_ladder")


# ============================================================
# Figure 3 - tier composition and version-record depth
# ============================================================

fig, (ax_left, ax_right) = plt.subplots(
    1, 2, figsize=(10.2, 4.3)
)

# Left: share of the 107 by tier.
tier_counts = (
    selected.groupby(["Tier", "Risk Tier"], as_index=False)
    .size()
    .sort_values("Tier")
)

bottom = 0

for _, row in tier_counts.iterrows():
    ax_left.bar(
        [0],
        [row["size"]],
        bottom=[bottom],
        width=0.55,
        color=TIER_COLORS[row["Tier"]],
        edgecolor="white",
        linewidth=0.8,
        label=row["Risk Tier"],
    )

    ax_left.text(
        0,
        bottom + row["size"] / 2,
        f"{row['size']}  ({row['size'] / 107:.0%})",
        ha="center",
        va="center",
        color="white" if row["Tier"] != "Tier 5" else "black",
        fontsize=9,
    )

    bottom += row["size"]

ax_left.set_xticks([])
ax_left.set_xlim(-0.45, 1.75)
ax_left.set_ylabel("Unique package names")
ax_left.set_title("Composition of the selected set", pad=8)
ax_left.grid(axis="y", alpha=0.30, linewidth=0.6)
ax_left.set_axisbelow(True)
ax_left.legend(
    frameon=False,
    fontsize=8,
    loc="center left",
    bbox_to_anchor=(0.40, 0.5),
)

# Right: how many matching versions each selected package has.
depth = (
    selected["Version Records"]
    .value_counts()
    .sort_index()
)

bars = ax_right.bar(
    depth.index.astype(str),
    depth.values,
    color="#17365D",
    edgecolor="white",
    width=0.6,
)

for bar, value in zip(bars, depth.values):
    ax_right.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        str(int(value)),
        ha="center",
        fontsize=9,
    )

ax_right.set_xlabel("Matching version records per package")
ax_right.set_ylabel("Packages")
ax_right.set_title("Version depth per selected package", pad=8)
ax_right.set_ylim(0, depth.values.max() * 1.16)
ax_right.grid(axis="y", alpha=0.30, linewidth=0.6)
ax_right.set_axisbelow(True)

fig.tight_layout()
save(fig, "fig3_composition_and_depth")

print(f"\nFigures written to: {FIGURE_FOLDER.resolve()}")

saved: fig1_category_frequency.png / fig1_category_frequency.pdf
saved: fig2_selection_ladder.png / fig2_selection_ladder.pdf
saved: fig3_composition_and_depth.png / fig3_composition_and_depth.pdf

Figures written to: D:\12. RQ2 eDySec\Adversarial Attacks\figures_107
